# traxr quickstart

Point **your own agent** at **your own data**, run controlled-perturbation experiments, and get back contamination/divergence metrics (`d_norm`, `t*`, manifestation, token overhead).

This notebook runs top-to-bottom **without an API key** — the real-model cells at the end are skip-safe.

In [ ]:
# Install (no-op when traxr is already importable, e.g. in CI)
try:
    import traxr
except ImportError:
    %pip install -q "traxr[document,openai,pandas,langgraph,viz] @ git+https://github.com/anna-mazhar/traxr.git@main"
    import traxr
print("traxr", traxr.__version__)

## 1. A no-key experiment

The bundled reference agent + the deterministic LLM stub run a complete experiment offline. First, some data:

In [ ]:
import tempfile
from pathlib import Path

workdir = Path(tempfile.mkdtemp(prefix="traxr-demo-"))
csv_path = workdir / "sales.csv"
csv_path.write_text("region,revenue\nnorth,10\nsouth,32\n")
print(csv_path.read_text())

A **dry run** prints the full plan with zero LLM calls and zero agent invocations:

In [ ]:
import warnings

import traxr

experiment = traxr.Experiment(
    files=csv_path,
    question="What is the total revenue?",
    expected_answer="42",
    llm=traxr.DeterministicLLMStub(scenario="identity", final_answer="42"),
)
plan = experiment.run(dry_run=True)

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    results = experiment.run()
print(results.summary())

## 2. What would run on *your* files?

The permutation matrix is agent-kind-aware (PDFs get surgical in-place operators for external agents, plus injection-only operators for the built-in agent):

In [ ]:
from traxr.data.sources import DataSource
from traxr.perturb.matrix import AgentKind, MatrixConfig, build_matrix

source = DataSource.from_path(csv_path)
for spec in build_matrix(source, MatrixConfig(agent_kind=AgentKind.EXTERNAL)):
    print(f"{spec.perturbation.value:20s} via {spec.delivery.value}")

## 3. Explore the results

In [ ]:
columns = ["perturbation", "d_norm", "t_star", "manifestation", "answer_changed", "task_success"]
results.to_dataframe()[columns]

In [ ]:
import matplotlib.pyplot as plt

from traxr import viz

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
viz.plot_d_norm(results, ax=axes[0])
viz.plot_t_star(results, ax=axes[1])
viz.plot_manifestations(results, ax=axes[2])
fig.tight_layout()

## 4. Bring your own agent (needs `OPENAI_API_KEY`)

Your agent is any `(Task) -> str` callable. Wrap its client with `traxr.instrument()` — that's the whole integration. **Cost note:** this runs 1 baseline + 1 perturbed + 1 noise-floor call on your key; `max_llm_calls_per_run` bounds runaway agents.

In [ ]:
import os

if not os.environ.get("OPENAI_API_KEY"):
    print("OPENAI_API_KEY not set — skipping the real-model cell.")
else:
    import openai

    client = traxr.instrument(openai.OpenAI())

    def my_agent(task: traxr.Task) -> str:
        data = task.files[0].read_text()
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": f"{task.question}\n\n{data}"}],
        )
        return response.choices[0].message.content or ""

    byo = traxr.Experiment(
        files=csv_path,
        question="What is the total revenue? Answer with just the number.",
        expected_answer="42",
        agent=my_agent,
        config=traxr.ExperimentConfig(
            perturbations=[traxr.PerturbationType.LABEL_CORRUPT],
            max_llm_calls_per_run=5,
        ),
    )
    print(byo.run().summary())

## 5. Bring your LangGraph (keyless demo)

`traxr.from_langgraph()` captures node transitions, tool calls, and LLM calls via callbacks. Here with a fake chat model, so it runs offline — swap in your real graph + model directly.

In [ ]:
try:
    from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
    from langchain_core.messages import AIMessage
    from langgraph.graph import END, START, MessagesState, StateGraph
except ImportError:
    print("pip install 'traxr[langgraph]' to run this cell — skipped.")
else:

    def build_graph_agent():
        model = GenericFakeChatModel(
            messages=iter(
                [
                    AIMessage(content="plan made"),
                    AIMessage(content="the answer is 42"),
                ]
            )
        )
        graph = StateGraph(MessagesState)
        graph.add_node("planner", lambda s: {"messages": [model.invoke(s["messages"])]})
        graph.add_node("analyst", lambda s: {"messages": [model.invoke(s["messages"])]})
        graph.add_edge(START, "planner")
        graph.add_edge("planner", "analyst")
        graph.add_edge("analyst", END)
        return traxr.from_langgraph(graph.compile())

    lg = traxr.Experiment(
        files=csv_path,
        question="What is the total revenue?",
        agent_factory=build_graph_agent,  # fresh graph state per run
        config=traxr.ExperimentConfig(
            perturbations=[traxr.PerturbationType.COLUMN_SWAP],
        ),
    )
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        lg_results = lg.run()
    print(lg_results.summary())

## 6. Custom scoring: LLM-as-a-judge (needs `OPENAI_API_KEY`)

The default scorer (`check_answer_match`) is exact normalized string equality — deliberately strict. A real agent answers in full sentences, so a bare `expected_answer="42"` will rarely literally equal the reply. Bring your own scorer via `ExperimentConfig(scorer=...)`; the built-in `llm_judge_match` asks an LLM whether the candidate answer reaches the same core conclusion. **Not deterministic** — it costs an extra LLM call per scored answer and can vary between runs.

In [ ]:
if not os.environ.get("OPENAI_API_KEY"):
    print("OPENAI_API_KEY not set — skipping the real-model cell.")
else:
    from traxr.scoring import llm_judge_match

    judged = traxr.Experiment(
        files=csv_path,
        question="What is the total revenue?",  # no "answer with just the number" hint this time
        expected_answer="42",
        agent=my_agent,
        config=traxr.ExperimentConfig(
            perturbations=[traxr.PerturbationType.LABEL_CORRUPT],
            max_llm_calls_per_run=5,
            scorer=llm_judge_match,
        ),
    )
    print(judged.run().summary())

## Where next

- The [README](https://github.com/anna-mazhar/traxr#readme): operator catalog, *is my agent traceable?*, metric definitions.
- [SECURITY.md](https://github.com/anna-mazhar/traxr/blob/main/SECURITY.md): read this before running an agent with real tools against perturbed data.
- `traxr operators` / `traxr selfcheck` on the command line.